<a href="https://colab.research.google.com/github/Joseonggwan/machine-learning-practice/blob/main/09_movie_recommandation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Movie Lens Dataset Recommandation

600명의 사용자가 9,000편의 영화에 10만 건의 평점과 3,600건의 태그를 적용한 데이터셋

- rating (이번 추천 알고리즘에 사용)
- movies
- links
- tags

## 라이브러리 불러오기

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
import numpy as np

## 데이터 불러오기 및 확인

In [4]:
# 평점
ratings = pd.read_csv('/content/drive/MyDrive/rating.csv')

ratings_with_datetime = ratings.copy()

ratings_with_datetime['date'] = pd.to_datetime(
    ratings_with_datetime['timestamp']
)

ratings_with_datetime.head()

,userId,movieId,rating,timestamp,date
0,1,2,3.5,2005.4.2 23:53,2005-04-02 23:53:00
1,1,29,3.5,2005.4.2 23:31,2005-04-02 23:31:00
2,1,32,3.5,2005.4.2 23:33,2005-04-02 23:33:00
3,1,47,3.5,2005.4.2 23:32,2005-04-02 23:32:00
4,1,50,3.5,2005.4.2 23:29,2005-04-02 23:29:00


In [8]:
tags = pd.read_csv('/content/drive/MyDrive/tag.csv', encoding='euc-kr')

tags_with_datetime = tags.copy()

tags_with_datetime['date'] = pd.to_datetime(
    tags_with_datetime['timestamp']
)

tags_with_datetime.head()

,userId,movieId,tag,timestamp,date
0,18,4141,Mark Waters,2009.4.24 18:19,2009-04-24 18:19:00
1,65,208,dark hero,2013.5.10 1:41,2013-05-10 01:41:00
2,65,353,dark hero,2013.5.10 1:41,2013-05-10 01:41:00
3,65,521,noir thriller,2013.5.10 1:39,2013-05-10 01:39:00
4,65,592,dark hero,2013.5.10 1:41,2013-05-10 01:41:00


In [9]:
links = pd.read_csv('/content/drive/MyDrive/link.csv')
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [11]:
movies = pd.read_csv('/content/drive/MyDrive/movie.csv', encoding='euc-kr')

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [12]:
#movieId = 60756 이거나 89774인 movies data 출력
movie_id = [60756, 89774]
movies[movies['movieId'].isin(movie_id)]

,movieId,title,genres
12867,60756,Step Brothers (2008),Comedy
17883,89774,Warrior (2011),Drama


In [13]:
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)
train_data

,userId,movieId,rating,timestamp
408561,2772,70159,4.0,2011.2.7 11:13
70143,489,2273,4.0,2002.7.8 16:08
708782,4710,344,4.5,2006.1.22 1:38
572694,3858,45,4.0,2010.3.9 13:25
774181,5155,17,2.5,2005.2.14 16:59
...,...,...,...,...
259178,1783,2422,0.5,2008.6.23 20:30
365838,2483,350,4.5,2003.8.24 18:08
131932,902,2871,1.0,2011.5.9 19:21
671155,4458,2605,1.0,2009.5.28 14:29


In [14]:
# 사용자-아이템 행렬 생성
train_matrix = train_data.pivot(index='userId', columns='movieId', values='rating')
test_matrix = test_data.pivot(index='userId', columns='movieId', values='rating')
train_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,129235,129303,129350,129354,129428,129707,130073,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7116,4.0,NaN,NaN,NaN,3.5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7117,4.0,NaN,4.0,NaN,NaN,5.0,3.0,NaN,1.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 결측값 처리
- 평점이 없는 경우 0으로 처리

In [15]:
train_matrix = train_matrix.fillna(0)
test_matrix = test_matrix.fillna(0)
train_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,129235,129303,129350,129354,129428,129707,130073,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7116,4.0,0.0,0.0,0.0,3.5,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7117,4.0,0.0,4.0,0.0,0.0,5.0,3.0,0.0,1.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 사용자 기반 협업 필터링

In [16]:
# 사용자 유사도
user_similarity = cosine_similarity(train_matrix)  # 사용자 간 코사인 유사도 계산
user_similarity = pd.DataFrame(user_similarity, index=train_matrix.index, columns=train_matrix.index)
user_similarity

userId,1,2,3,4,5,6,7,8,9,10,...,7111,7112,7113,7114,7115,7116,7117,7118,7119,7120
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.086213,0.194995,0.018328,0.118393,0.000000,0.125351,0.055358,0.078156,0.109049,...,0.102751,0.000000,0.035603,0.062323,0.055043,0.137699,0.160225,0.058099,0.045021,0.053969
2,0.086213,1.000000,0.131097,0.039998,0.103819,0.049259,0.103033,0.075819,0.000000,0.031883,...,0.014410,0.000000,0.079983,0.000000,0.032433,0.081670,0.090655,0.090210,0.091234,0.040078
3,0.194995,0.131097,1.000000,0.051826,0.114537,0.022693,0.156921,0.090010,0.037816,0.151474,...,0.040010,0.051850,0.111771,0.078478,0.108605,0.169573,0.195568,0.149144,0.164081,0.000000
4,0.018328,0.039998,0.051826,1.000000,0.267719,0.049592,0.090991,0.258353,0.053423,0.032099,...,0.092399,0.064460,0.404408,0.000000,0.286214,0.087354,0.242209,0.000000,0.244933,0.000000
5,0.118393,0.103819,0.114537,0.267719,1.000000,0.151756,0.157243,0.177538,0.000000,0.058467,...,0.070781,0.000000,0.295298,0.000000,0.226888,0.128582,0.266624,0.033352,0.293421,0.080843
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7116,0.137699,0.081670,0.169573,0.087354,0.128582,0.043032,0.172095,0.137881,0.035310,0.104882,...,0.058446,0.000000,0.123731,0.088166,0.128548,1.000000,0.211501,0.067922,0.142539,0.038137
7117,0.160225,0.090655,0.195568,0.242209,0.266624,0.144102,0.160223,0.312566,0.031435,0.118453,...,0.121949,0.032310,0.332564,0.131068,0.283233,0.211501,1.000000,0.068103,0.247182,0.066996
7118,0.058099,0.090210,0.149144,0.000000,0.033352,0.000000,0.052132,0.035599,0.000000,0.060328,...,0.000000,0.058621,0.000000,0.118505,0.021996,0.067922,0.068103,1.000000,0.042836,0.000000


In [17]:
def predict_user_based_cf(user_id, movie_id):
    # 유사도와 평점을 기반으로 예측
    if movie_id not in train_matrix.columns:
        return 0  # 영화가 훈련 데이터에 없으면 0 반환
    similar_users = user_similarity[user_id]
    ratings = train_matrix[movie_id]
    weighted_sum = np.dot(similar_users, ratings)
    similarity_sum = np.sum(similar_users[ratings > 0])
    if similarity_sum == 0:
        return 0
    return weighted_sum / similarity_sum

In [18]:
user_based_predictions = []
for user_id, movie_id, rating in test_data[['userId', 'movieId', 'rating']].values:
    pred = predict_user_based_cf(user_id, movie_id)
    user_based_predictions.append(pred)

In [24]:
from re import T
target_movie_data = train_matrix.iloc[:, int(movie_id)]
series = target_movie_data[target_movie_data > 0]

print(series)

userId
554     4.0
982     3.5
1037    3.0
1296    4.0
1755    5.0
2537    4.0
2710    3.0
2931    3.5
3197    4.0
3581    3.0
3858    1.5
4417    4.5
5518    4.0
5698    2.0
6064    4.0
6081    1.0
6424    4.0
6539    3.0
6800    3.0
6976    3.0
Name: 565, dtype: float64


user_id 103번과 495는 복싱영화의 평점을 높게 평가

In [26]:
user_id = 103                  # 사용자 ID
pred_movie_id = movie_id       # 영화 ID 60756

predicted_rating = predict_user_based_cf(user_id, pred_movie_id)

print(f"{user_id}번 사용자의 영화번호 {pred_movie_id}번의 평점 예측: {predicted_rating:.2f}")

103번 사용자의 영화번호 551.0번의 평점 예측: 3.81


In [27]:
user_based_rmse = np.sqrt(mean_squared_error(test_data['rating'], user_based_predictions))
print(f"사용자 기반 협업 필터링 RMSE: {user_based_rmse:.4f}")

사용자 기반 협업 필터링 RMSE: 0.9551


## 아이템 기반 협업 필터링

In [28]:
item_similarity = cosine_similarity(train_matrix.T)  # 아이템 간 코사인 유사도 계산
item_similarity = pd.DataFrame(item_similarity, index=train_matrix.columns, columns=train_matrix.columns)
item_similarity

movieId,1,2,3,4,5,6,7,8,9,10,...,129235,129303,129350,129354,129428,129707,130073,130462,130490,130642
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.333783,0.249603,0.116546,0.276251,0.314578,0.256537,0.081633,0.119533,0.316995,...,0.021535,0.018843,0.024226,0.021535,0.027445,0.0,0.016151,0.024226,0.025513,0.021535
2,0.333783,1.000000,0.171931,0.117337,0.219002,0.220346,0.187290,0.138154,0.114501,0.346709,...,0.000000,0.019393,0.038786,0.038786,0.000000,0.0,0.029090,0.000000,0.018647,0.000000
3,0.249603,0.171931,1.000000,0.103715,0.332801,0.213970,0.276728,0.060327,0.172284,0.151033,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
4,0.116546,0.117337,0.103715,1.000000,0.126338,0.115017,0.177988,0.154431,0.138601,0.116165,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
5,0.276251,0.219002,0.332801,0.126338,1.000000,0.211321,0.331388,0.120416,0.188639,0.172079,...,0.000000,0.026587,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.025564,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129707,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
130073,0.016151,0.029090,0.000000,0.000000,0.000000,0.000000,0.029890,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,0.000000,0.000000,0.000000
130462,0.024226,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,0.000000


In [29]:
def predict_item_based(user_id, movie_id):
    # 유사도와 평점을 기반으로 예측
    if movie_id not in train_matrix.columns:
        return 0  # 영화가 훈련 데이터에 없으면 0 반환
    similar_items = item_similarity[movie_id]
    user_ratings = train_matrix.loc[user_id]
    weighted_sum = np.dot(similar_items, user_ratings)
    similarity_sum = np.sum(similar_items[user_ratings > 0])
    if similarity_sum == 0:
        return 0
    return weighted_sum / similarity_sum

In [30]:
train_matrix.loc[user_id]

,103
movieId,
1,0.0
2,0.0
3,0.0
4,0.0
5,0.0
...,...
129707,0.0
130073,0.0
130462,0.0


In [31]:
item_based_predictions = []
for user_id, movie_id, rating in test_data[['userId', 'movieId', 'rating']].values:
    pred = predict_item_based(user_id, pred_movie_id)
    item_based_predictions.append(pred)

In [33]:
print(f"{user_id}번 사용자의 영화번호 {pred_movie_id}번의 평점 예측: {predicted_rating:.2f}")

103번 사용자의 영화번호 89774번의 평점 예측: 3.79


In [34]:
item_based_rmse = np.sqrt(mean_squared_error(test_data['rating'], item_based_predictions))
print(f"아이템 기반 협업 필터링 RMSE: {item_based_rmse:.4f}")

아이템 기반 협업 필터링 RMSE: 0.9721


## 성능 비교

In [35]:
print(f"사용자 기반 협업 필터링 RMSE: {user_based_rmse:.4f}")
print(f"아이템 기반 협업 필터링 RMSE: {item_based_rmse:.4f}")

사용자 기반 협업 필터링 RMSE: 0.9551
아이템 기반 협업 필터링 RMSE: 0.9721


In [32]:
user_id = 103
pred_movie_id = 89774

predicted_rating = predict_user_based_cf(user_id, pred_movie_id)

print(f"{user_id}번 사용자의 영화번호 {pred_movie_id}번의 평점 예측: {predicted_rating:.2f}")

103번 사용자의 영화번호 89774번의 평점 예측: 3.79


확인문제 1 : 60756번 영화를 본 사용자 중에 1명을 찾아서 해당 사용자의 89774번 영화의 평점을 예측하세요
- 대략 4점

확인문제 2 : 위 성능 비교 했을 때 사용자 기반과 아이템 기반 중 더 좋은 성능을 나타내는 방법을 쓰세요
- 사용자 기반이 더 좋은 성능을 나타낸다.